## Payments data validation and transformation

In [0]:
%sql
SELECT * FROM course_training_catalog.bronze_olist.order_payments LIMIT 10;

### if Data profile doesn't load in the free edition, we can do the following quality checks in SQL for any missing values or duplicate records


In [0]:
%sql
-- check for missing values in all variables of the data set
SELECT 
  COUNT(*) AS total_rows,
  SUM(CASE WHEN order_id is NULL THEN 1 ELSE 0 END) AS n_null_order_id,
  SUM(CASE WHEN payment_sequential is NULL THEN 1 ELSE 0 END) AS n_null_payment_sequential,
  SUM(CASE WHEN payment_type is NULL THEN 1 ELSE 0 END) AS n_null_payment_type,
  SUM(CASE WHEN payment_installments is NULL THEN 1 ELSE 0 END) AS n_null_payment_installments,
  SUM(CASE WHEN payment_value is NULL THEN 1 ELSE 0 END) AS n_null_payment_value
FROM course_training_catalog.bronze_olist.order_payments


In [0]:
%sql 
-- Check for duplicates
SELECT order_id, payment_sequential, COUNT(*) AS cnt
FROM course_training_catalog.bronze_olist.order_payments
GROUP BY order_id, payment_sequential
HAVING COUNT(*) > 1
ORDER BY cnt DESC;


Note that the above check for duplicates didn't produce any results, that means there are no data duplicate issues. 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payments_norm AS
SELECT 
  order_id,
  TRY_CAST(payment_sequential AS INTEGER) AS payment_sequential,
  LOWER(TRIM(payment_type)) AS payment_type,
  TRY_CAST(payment_installments AS INTEGER) AS payment_installments,
  CAST(ROUND(TRY_CAST(payment_value AS DOUBLE), 2) AS DECIMAL (18,2)) AS payment_value
  -- note that we used try_cast for double and then used 'cast'; becasue if we use cast directly, 
  -- the system tries to convert each value straight into a fixed point format like 18 comma two.
  -- that's a very strict data type and even the smallest or slightest formatting issue can result in an
  -- error.so, for better error tolerance and conversion performance, we used try_cast first and then
  -- cast to a fixed point format.
FROM course_training_catalog.bronze_olist.order_payments
    


In [0]:
%sql
SELECT * FROM payments_norm LIMIT 10;

To summarize all of this; in the silver layer we achieved
- schema consistency
- correct data types
- standardization
- lower plus trim
- financial accuracy
- round plus decimal
- fault tolerant processing
- try, cast
- reusability
- Temp view plus or replace

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW payments_flagged AS
SELECT
 *,
 CASE WHEN payment_installments > 0 AND payment_value IS NOT null
 THEN ROUND(payment_value/payment_installments, 2)
 ELSE NULL END AS payment_per_installment
 FROM payments_norm;

In [0]:
%sql
SELECT * FROM payments_flagged LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.silver_olist.order_payments
USING DELTA
AS
SELECT
  order_id,
  payment_sequential,
  payment_type,
  payment_installments,
  payment_value,
  payment_per_installment
FROM payments_flagged;

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.order_payments LIMIT 10;

### Building the silver version of order_reviews data
plan for handling this data in silver layer
- each column has the correct data type
- basic text normalization - trimming and reducing multiple spaces into one
- handle missing or incorrect data (for isntance many cusotmers only give a score without writing a comment, so review comments might have null values), we will flag them instead of filling them
- we will also move severely corrupted rows; for e.g, those with failed data conversions or missing reveiw IDs to a quarantine table. this way we can prseverve problematic records without contaminating the main table. 

In [0]:
%sql
SELECT * FROM course_training_catalog.bronze_olist.order_reviews LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW reviews_stage AS
SELECT
  TRIM(review_id) AS review_id,
  TRIM(order_id) AS order_id,
  TRY_CAST(review_score AS INT) As review_score,
  review_creation_date AS review_creation_str,
  review_answer_timestamp AS review_answer_str,

  TRY_CAST(review_creation_date AS TIMESTAMP) AS review_creation_ts,
  TRY_CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_ts,

  REGEXP_REPLACE(TRIM(review_comment_title), '\\s+', ' ') AS review_comment_title,
  REGEXP_REPLACE(TRIM(review_comment_message), '\\s+', ' ') AS review_comment_message
FROM course_training_catalog.bronze_olist.order_reviews;

SELECT * FROM reviews_stage LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.silver_olist.reviews_bad_rows AS
SELECT
  *
  FROM reviews_stage
  WHERE review_id is NULL
  OR order_id is NULL
  OR review_score is NULL
  OR (review_creation_ts IS NULL AND COALESCE(TRIM(review_creation_str), '') <> '')
  OR (review_answer_ts IS NULL AND COALESCE(TRIM(review_answer_str), '') <> '');

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.reviews_bad_rows;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW reviews_valid_dedup AS
SELECT *
FROM (
  SELECT 
    s.*,
    ROW_NUMBER() OVER(
      PARTITION BY s.review_id
      ORDER BY s.review_answer_ts DESC NULLS LAST, s.review_creation_ts DESC NULLS LAST
    ) AS rn
  FROM reviews_stage s
  WHERE s.review_id IS NOT NULL AND s.review_score BETWEEN 1 AND 5
) t -- this 't' is an alias for the inner query which will be referenced later
WHERE rn = 1;
   

In [0]:
%sql
SELECT * FROM reviews_valid_dedup LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW reviews_enriched AS
SELECT 
  review_id,
  order_id,
  review_score,
  review_creation_ts,
  review_answer_ts,
  CASE 
    WHEN review_comment_title IS NULL AND review_comment_message IS NOT NULL THEN SUBSTRING(review_comment_message, 1, 50)
    ELSE review_comment_title 
  END AS review_comment_title,

  CASE WHEN review_comment_title IS NULL AND review_comment_message IS NULL THEN TRUE ELSE FALSE END AS is_missing_comment,

  CASE WHEN review_comment_title IS NOT NULL AND review_comment_message IS NULL THEN TRUE ELSE FALSE END AS has_title_only,

  CASE WHEN review_comment_title IS NULL AND review_comment_message IS NOT NULL THEN TRUE ELSE FALSE END AS has_message_only

  FROM reviews_valid_dedup;



In [0]:
%sql
SELECT * FROM reviews_enriched LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.silver_olist.order_reviews AS
SELECT * FROM reviews_enriched;

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.order_reviews LIMIT 10;

In [0]:
%sql
COMMENT ON TABLE course_training_catalog.silver_olist.order_reviews IS
  'Order reviews (Silver): typed, deduplicated, normalized, with quality flags. 1 row = 1 review_id'

In [0]:
%sql
COMMENT ON COLUMN course_training_catalog.silver_olist.order_reviews.review_score IS
  'customer rating (1...5)'

In [0]:
%sql
ALTER TABLE course_training_catalog.silver_olist.order_reviews
ADD CONSTRAINT chk_review_score_1_5 CHECK (review_score BETWEEN 1 AND 5);


In [0]:
%sql
USE CATALOG course_training_catalog

In [0]:
%sql
DESCRIBE EXTENDED course_training_catalog.silver_olist.order_reviews

### Working on Geo Location data set

In [0]:
%sql
SELECT * FROM course_training_catalog.bronze_olist.geolocation LIMIT 10;

### Action plan for geo location table
- data type conversions converting latitude and longitude values to numeric types, city and state names to text 
- string normalization - trimming unnecessary spaces, converting all city names to lowercase and state codes to uppper case. 
- also simplify special characters so that name like Sao Paulo (someitme a special symbeol can be used for 'a' in Sao) become consistent across the data set. 
- Next will be deduplication, keeping one pne record per zip prefix
- we will use both python and sql here

In [0]:
geo_base = spark.table("course_training_catalog.bronze_olist.geolocation")
geo_base.createOrReplaceTempView("geo_base")

In [0]:
%sql
SELECT * FROM geo_base;

In [0]:
from pyspark.sql import functions as F
geo_typed = (geo_base
  .withColumn("zip_prefix", F.col("geolocation_zip_code_prefix").cast("integer"))
  .withColumn("lat", F.col("geolocation_lat").cast("double"))
  .withColumn("lng", F.col("geolocation_lng").cast("double"))
  .withColumnRenamed("geolocation_city", "city_raw")
  .withColumnRenamed("geolocation_state", "state_raw")
)

geo_typed.createOrReplaceTempView("geo_typed")

In [0]:
%sql
SELECT * FROM geo_typed;

In [0]:
%sql
-- create normalize values
CREATE OR REPLACE TEMP VIEW geo_norm AS
SELECT
  zip_prefix,
  lat,
  lng,
  lower(
      regexp_replace(
        translate(trim(regexp_replace(city_raw, '\\s+', ' ')),
        'àáâäǎeèéêëěiìíîïǐoòóôöǒuùúûüǔÀÁÂÄǍEÈÉÊËĚIÌÍÎÏǏOÒÓÔÖǑUÙÚÛÜǓ',
        'aaaaaeeeeiiiiooooouuuuuuAAAAAEEEEEIIIOOOOOOUUUUUU'
        ),
        '[^a-z]',' '
      )
  ) AS city_lc,
  upper(trim(regexp_replace(state_raw, '\\s+', ' '))) AS state_uc
FROM geo_typed;

In [0]:
%sql
SELECT * FROM geo_norm LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
  COUNT(DISTINCT zip_prefix) AS distinct_zip
  FROM geo_norm;

In [0]:
%sql
SELECT zip_prefix, COUNT(*) AS n
FROM geo_norm
GROUP BY zip_prefix
ORDER BY n DESC
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW mode_city AS
SELECT
  zip_prefix,
  city_lc AS city,
  state_uc AS state
FROM (
  SELECT
    zip_prefix,
    city_lc,
    state_uc,
    COUNT(*) AS c,
    ROW_NUMBER() OVER(
      PARTITION BY zip_prefix
      ORDER BY COUNT(*) DESC, city_lc ASC
    ) AS rn
  FROM geo_norm
  GROUP BY zip_prefix, city_lc, state_uc
)
WHERE rn = 1;

In [0]:
%sql
SELECT COUNT(*) AS total_rows,
  COUNT(DISTINCT zip_prefix) AS distinct_zip
FROM mode_city;

In [0]:
%sql
SELECT zip_prefix, COUNT(*) AS n
FROM mode_city
GROUP BY zip_prefix
ORDER BY n DESC
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW centroids AS
SELECT
  zip_prefix,
  AVG(lat) AS lat,
  AVG(lng) AS lng
FROM geo_norm
GROUP BY zip_prefix;
    
SELECT * FROM centroids LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW geo_dedup AS
SELECT
  c.zip_prefix,
  c.lat,
  c.lng,
  m.city,
  m.state
FROM centroids AS c
LEFT JOIN mode_city AS m
USING(zip_prefix);
    


In [0]:
%sql
SELECT * FROM geo_dedup LIMIT 10;


In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.silver_olist.geolocation AS
SELECT
  zip_prefix AS geolocation_zip_code_prefix,
  ROUND(lat, 6) As geolocation_lat,
  ROUND(lng, 6) AS geolocation_lng,
  city AS geolocation_city,
  state AS geolocation_state
FROM geo_dedup;

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.geolocation LIMIT 10;

### Clean Reference tables in the silver layer: product_category_name_translation_table

In [0]:
%sql
SELECT * FROM course_training_catalog.bronze_olist.product_category_name_translation

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.silver_olist.category_lookup AS
SELECT
  product_category_name AS category_pt,
  product_category_name_english AS category_en
FROM course_training_catalog.bronze_olist.product_category_name_translation;
    
SELECT * FROM course_training_catalog.silver_olist.category_lookup